# Notebook 4 — AFM Scanning Simulator (Raster Scan, Feedback, Imaging Artifacts)

This notebook simulates a **basic AFM raster scan** and illustrates how **feedback bandwidth**, **scan speed**, and **noise** create familiar imaging artifacts (e.g., rounded edges, streaking/lag, overshoot).

**Learning goals**
- Understand raster scanning (fast/slow axis) and pixel dwell time.
- See how limited feedback bandwidth creates *tracking error*.
- Explore artifacts: edge rounding, phase/lag, overshoot, noise, drift.
- Connect **Chapter 3 (instrumentation + feedback)** to **imaging modes**.

> This is a *didactic simulator*: it is not a full instrument model, but it captures the key ideas with minimal math and maximum intuition.


## Model overview (conceptual)

We generate a synthetic surface height map \( z(x,y) \) (steps, bumps, roughness).  
During scanning, the AFM tries to keep a target interaction constant by adjusting the **Z-piezo command** \( z_p \).

A simple way to model feedback limits is a **first-order closed-loop response**. In this notebook we implement a practical discrete-time model with an effective bandwidth limit.

### What you will plot
- **True surface** (ground truth)
- **Measured image** (what AFM would report under given scan + feedback conditions)
- **Tracking error** \( z - z_p \) (where artifacts come from)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Widgets for Colab/Jupyter
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

np.random.seed(0)


## 1) Generate a synthetic surface

Choose a surface type and parameters. We generate \(z(x,y)\) on a grid.


In [ ]:
def make_surface(nx=256, ny=256, kind="Step + bumps", amp=20e-9, rough=2e-9):
    """Return (X,Y,Z) with Z in meters."""
    x = np.linspace(0, 1.0, nx)
    y = np.linspace(0, 1.0, ny)
    X, Y = np.meshgrid(x, y, indexing="xy")
    Z = np.zeros_like(X)

    if kind == "Step + bumps":
        Z += amp * (X > 0.5)  # step
        for (x0, y0, a, s) in [(0.25, 0.30, 0.7*amp, 0.06), (0.72, 0.65, -0.5*amp, 0.08)]:
            Z += a * np.exp(-((X-x0)**2 + (Y-y0)**2)/(2*s**2))
    elif kind == "Sinusoidal grating":
        Z += amp * np.sin(2*np.pi*6*X)
    elif kind == "Random roughness":
        Z += rough * np.random.randn(*Z.shape)
    elif kind == "Pillars":
        for (x0, y0) in [(0.25,0.25),(0.75,0.25),(0.25,0.75),(0.75,0.75)]:
            Z += amp * np.exp(-((X-x0)**2 + (Y-y0)**2)/(2*(0.03)**2))
    else:
        raise ValueError("Unknown surface kind")

    if rough > 0 and kind != "Random roughness":
        Z += rough * np.random.randn(*Z.shape)

    Z -= Z.mean()
    return X, Y, Z

def plot_map(Z, title="Map", nm=True):
    plt.figure(figsize=(6,5))
    img = Z*1e9 if nm else Z
    plt.imshow(img, origin="lower", aspect="equal")
    plt.colorbar(label=("Height (nm)" if nm else "Height (m)"))
    plt.title(title)
    plt.xlabel("x pixels")
    plt.ylabel("y pixels")
    plt.tight_layout()
    plt.show()


## 2) Raster scan + simplified feedback

We simulate line-by-line scanning. The “measured” image is the **Z command** produced by the feedback loop while trying to track the surface.

Key parameters:
- **Scan rate (lines/s)** and **pixels/line** set the dwell time per pixel.
- **Feedback bandwidth (Hz)** limits how fast \(z_p\) can follow \(z\).
- **PI gains** (dimensionless in this simplified model) influence lag vs overshoot.


In [ ]:
def raster_path(nx, ny, bidirectional=True):
    """Return list of (iy, ix_sequence) for raster scanning."""
    rows = []
    for iy in range(ny):
        if bidirectional and (iy % 2 == 1):
            rows.append((iy, np.arange(nx-1, -1, -1)))
        else:
            rows.append((iy, np.arange(nx)))
    return rows

def simulate_scan(Z_true,
                  line_rate=1.0,           # lines per second
                  bidirectional=True,
                  fb_bw=200.0,             # feedback bandwidth (Hz), effective
                  Kp=0.6, Ki=0.05,         # PI-like gains (dimensionless, didactic)
                  noise_rms=0.5e-9,        # meters
                  drift_per_line=0.0e-9,   # meters/line
                  tip_radius_px=0,         # simple blur radius (pixels)
                  seed=0):
    """Return Z_meas (piezo command) and error map."""
    rng = np.random.default_rng(seed)
    ny, nx = Z_true.shape

    # Optional proxy for tip convolution: blur by a disk kernel (didactic)
    Z = Z_true.copy()
    if tip_radius_px and tip_radius_px > 0:
        r = int(tip_radius_px)
        yy, xx = np.ogrid[-r:r+1, -r:r+1]
        mask = (xx*xx + yy*yy) <= r*r
        kernel = mask.astype(float)
        kernel /= kernel.sum()

        Zf = np.fft.rfft2(Z)
        k = np.zeros_like(Z)
        k[:kernel.shape[0], :kernel.shape[1]] = kernel
        k = np.roll(k, -r, axis=0)
        k = np.roll(k, -r, axis=1)
        Kf = np.fft.rfft2(k)
        Z = np.fft.irfft2(Zf * Kf, s=Z.shape)

    # Time step per pixel
    dt_line = 1.0 / max(line_rate, 1e-9)   # seconds per line
    dt = dt_line / nx                      # seconds per pixel

    # Feedback time constant from bandwidth (1st order approximation)
    tau = 1.0 / (2*np.pi*max(fb_bw, 1e-9))

    Z_meas = np.zeros_like(Z)
    err_map = np.zeros_like(Z)

    z_p = 0.0     # piezo command (m)
    integ = 0.0   # integrator state

    rows = raster_path(nx, ny, bidirectional=bidirectional)

    for iy, xs in rows:
        z_p += drift_per_line  # slow drift per line

        for ix in xs:
            z = Z[iy, ix]
            z_noisy = z + rng.normal(0.0, noise_rms)

            e = (z_noisy - z_p)
            integ += e * dt
            u = Kp*e + Ki*integ

            # Bandwidth-limited response to control effort u
            z_p += (dt/tau) * u

            Z_meas[iy, ix] = z_p
            err_map[iy, ix] = (z - z_p)  # error vs true surface (no noise)

    return Z_meas, err_map


## 3) Interactive simulator (sliders)

Suggested explorations:

1. Increase **scan speed** at fixed bandwidth → more lag and edge rounding.  
2. Decrease **bandwidth** at fixed scan speed → similar degradation.  
3. Increase **Kp** too much → overshoot / ringing.  
4. Turn on **bidirectional** scan → compare forward/backward line artifacts.  
5. Add **drift** → line-to-line offsets.


In [ ]:
def run_demo(surface_kind="Step + bumps",
             amp_nm=20.0,
             rough_nm=2.0,
             nx=256,
             ny=256,
             line_rate=1.0,
             bidirectional=True,
             fb_bw=200.0,
             Kp=0.6,
             Ki=0.05,
             noise_nm=0.5,
             drift_nm_per_line=0.0,
             tip_radius_px=0,
             show_error=True):
    _, _, Z = make_surface(nx=nx, ny=ny, kind=surface_kind, amp=amp_nm*1e-9, rough=rough_nm*1e-9)

    Z_meas, err = simulate_scan(
        Z,
        line_rate=line_rate,
        bidirectional=bidirectional,
        fb_bw=fb_bw,
        Kp=Kp,
        Ki=Ki,
        noise_rms=noise_nm*1e-9,
        drift_per_line=drift_nm_per_line*1e-9,
        tip_radius_px=tip_radius_px,
        seed=0
    )

    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.imshow(Z*1e9, origin="lower", aspect="equal")
    plt.title("True surface (nm)")
    plt.colorbar()

    plt.subplot(1,2,2)
    plt.imshow(Z_meas*1e9, origin="lower", aspect="equal")
    plt.title("Measured image (Z command, nm)")
    plt.colorbar()

    plt.tight_layout()
    plt.show()

    if show_error:
        plt.figure(figsize=(6,5))
        plt.imshow(err*1e9, origin="lower", aspect="equal")
        plt.title("Tracking error (true z − z_p) (nm)")
        plt.colorbar()
        plt.tight_layout()
        plt.show()

if WIDGETS_AVAILABLE:
    interact(
        run_demo,
        surface_kind=Dropdown(options=["Step + bumps","Sinusoidal grating","Random roughness","Pillars"], value="Step + bumps"),
        amp_nm=FloatSlider(min=0.0, max=50.0, step=1.0, value=20.0),
        rough_nm=FloatSlider(min=0.0, max=10.0, step=0.5, value=2.0),
        nx=IntSlider(min=64, max=512, step=64, value=256),
        ny=IntSlider(min=64, max=512, step=64, value=256),
        line_rate=FloatSlider(min=0.2, max=10.0, step=0.2, value=1.0, description="lines/s"),
        bidirectional=Checkbox(value=True),
        fb_bw=FloatSlider(min=10.0, max=2000.0, step=10.0, value=200.0, description="BW (Hz)"),
        Kp=FloatSlider(min=0.0, max=2.0, step=0.05, value=0.6),
        Ki=FloatSlider(min=0.0, max=0.5, step=0.01, value=0.05),
        noise_nm=FloatSlider(min=0.0, max=5.0, step=0.1, value=0.5),
        drift_nm_per_line=FloatSlider(min=-2.0, max=2.0, step=0.05, value=0.0),
        tip_radius_px=IntSlider(min=0, max=10, step=1, value=0),
        show_error=Checkbox(value=True)
    )
else:
    print("ipywidgets not available here. Run in Google Colab for sliders.")
    run_demo()


## 4) Suggested exercises (for the textbook)

1. **Bandwidth vs scan speed:** Keep BW fixed and increase line rate until the step edge becomes visibly rounded. Explain why using pixel dwell time.
2. **Overshoot and ringing:** Increase \(K_p\) at fixed BW. When do you see overshoot? Relate this to feedback stability.
3. **Bidirectional mismatch:** Turn on bidirectional scanning and increase scan rate. Compare forward vs backward lines. Why can these differ in real instruments?
4. **Tip radius effect:** Increase `tip_radius_px` and explain how finite tip size modifies sharp features.
5. **Drift:** Add drift per line. How does it appear in the image? Which instrument subsystems contribute to drift in real AFM?


---
### Notes for instructors

- The feedback model is deliberately simplified. It is intended to build intuition for imaging artifacts that originate from **finite feedback bandwidth** and **finite scan speed**.
- You can extend this notebook by adding: hysteresis in XY scanner, creep, nonlinearity, and a more realistic control loop.
